In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [2]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

In [3]:
DB_USER='root'
DB_PASSWORD='OJlehXTRlEa6^zsFNILjnVoew9ku=E'
DB_NAME='essidb'
DB_PORT="5432"
DB_HOST='read-only-powerbi.cktgjnvajmd8.us-east-1.rds.amazonaws.com'
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

In [4]:
DB_USER='postgres'
DB_PASSWORD='AKindOfMagic'
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST='10.0.1.228'
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [5]:
DB_USER='postgres'
DB_PASSWORD='AKindOfMagic'
DB_NAME="indicadores_gctic"
DB_PORT="5432"
DB_HOST='10.0.1.228'
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [6]:
ipress_query = """SELECT ras.REDASISCOD, COUNT(DISTINCT cas.CENASICOD) AS CANTIDAD FROM CMCAS10 cas LEFT OUTER JOIN CMRAS10 ras ON cas.REDASISCOD = ras.REDASISCOD WHERE ORICENASICOD = '1' AND (TIPCENTASISCOD IN ('03','04','05','06','07','08')) AND ESTREGCOD = '1' GROUP BY ras.REDASISCOD"""
ipress_miconsulta_query = """SELECT ras.REDASISCOD, COUNT(DISTINCT t.CENASICOD) AS CANTIDAD
FROM CMSAS10 t 
LEFT OUTER JOIN cspar10 m ON m.paroricenasicod = t.oricenasicod 
AND m.parcenasicod = t.cenasicod 
LEFT OUTER JOIN CMCAS10 cas ON t.CENASICOD = cas.CENASICOD
LEFT OUTER JOIN CMRAS10 ras ON cas.REDASISCOD = ras.REDASISCOD
WHERE t.areseractsubciteelflg = '1' AND m.parciteel = '1'
GROUP BY ras.REDASISCOD"""
ipress = pd.read_sql_query(ipress_query, con=connection0)
ipress_miconsulta = pd.read_sql_query(ipress_miconsulta_query, con=connection0)

In [7]:
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
ipress=pd.merge(ipress,red,how='left', left_on='redasiscod', right_on='cod_red')
ipress=ipress.drop('redasiscod',axis=1)
ipress=ipress.drop('cod_red',axis=1)
ipress_miconsulta=pd.merge(ipress_miconsulta,red,how='left', left_on='redasiscod', right_on='cod_red')
ipress_miconsulta=ipress_miconsulta.drop('redasiscod',axis=1)
ipress_miconsulta=ipress_miconsulta.drop('cod_red',axis=1)

In [8]:
ipress.to_sql(name=f'ipress', con=connection3, if_exists='replace', index=False,chunksize=10)
ipress_miconsulta.to_sql(name=f'ipress_miconsulta', con=connection3, if_exists='replace', index=False,chunksize=10)

30

In [9]:
citas_medicas_query="""SELECT
    ras.REDASISCOD,
    cas.CENASICOD,
    c.MODOTORCITACOD,
    c.ESTCITCOD,
    c.CITAMBPROCONFEC AS FECHA,
    COUNT(*) AS CANTIDAD
FROM 
    CTCAM10 c
LEFT OUTER JOIN 
    CMCAS10 cas ON c.CITAMBCENASICOD = cas.CENASICOD
LEFT OUTER JOIN 
    CMRAS10 ras ON cas.REDASISCOD = ras.REDASISCOD
WHERE 
    c.CITAMBPROCONFEC >= TO_DATE('2023-01-01', 'YYYY-MM-DD')
GROUP BY 
    ras.REDASISCOD, cas.CENASICOD, c.CITAMBCENASICOD, c.MODOTORCITACOD, c.ESTCITCOD, c.CITAMBPROCONFEC
"""
citas_medicas = pd.read_sql_query(citas_medicas_query, con=connection0)

In [10]:
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas", con=connection2)
tipemi = pd.read_sql_query(f"SELECT id_tipemi,cod_emi FROM dim_tipemi", con=connection2)
eci = pd.read_sql_query(f"SELECT id_estcit,cod_eci FROM dim_estcit", con=connection2)

citas_medicas=pd.merge(citas_medicas,red,how='left', left_on='redasiscod', right_on='cod_red')
citas_medicas=citas_medicas.drop('redasiscod',axis=1)
citas_medicas=citas_medicas.drop('cod_red',axis=1)

citas_medicas=pd.merge(citas_medicas,cas,how='left', left_on='cenasicod', right_on='cod_cas')
citas_medicas=citas_medicas.drop('cenasicod',axis=1)
citas_medicas=citas_medicas.drop('cod_cas',axis=1)

citas_medicas=pd.merge(citas_medicas,tipemi,how='left', left_on='modotorcitacod', right_on='cod_emi')
citas_medicas=citas_medicas.drop('modotorcitacod',axis=1)
citas_medicas=citas_medicas.drop('cod_emi',axis=1)

citas_medicas=pd.merge(citas_medicas,eci,how='left', left_on='estcitcod', right_on='cod_eci')
citas_medicas=citas_medicas.drop('estcitcod',axis=1)
citas_medicas=citas_medicas.drop('cod_eci',axis=1)

In [11]:
citas_medicas.to_sql(name=f'citas_medicas', con=connection3, if_exists='replace', index=False,chunksize=10000)

97069

In [12]:
citas_miconsulta_query="""SELECT
    cen_asi_cod,
    DATE(date_create),
    COUNT(DISTINCT numero_doc_ident) AS total_personas
FROM
    cita
WHERE
    ind_confirmado = true
GROUP BY
    cen_asi_cod, DATE(date_create)
ORDER BY
    cen_asi_cod"""

citas_miconsulta = pd.read_sql_query(citas_miconsulta_query, con=connection1)

In [13]:
citas_miconsulta=pd.merge(citas_miconsulta,cas,how='left', left_on='cen_asi_cod', right_on='cod_cas')
citas_miconsulta=citas_miconsulta.drop('cen_asi_cod',axis=1)
citas_miconsulta=citas_miconsulta.drop('cod_cas',axis=1)

In [14]:
citas_miconsulta.to_sql(name=f'citas_miconsulta', con=connection3, if_exists='replace', index=False,chunksize=10000)

8719

In [15]:
usuarios_miconsulta_query="""SELECT
  EXTRACT(YEAR FROM u.date_create) AS año,
  EXTRACT(MONTH FROM u.date_create) AS mes,
  c.cod_red,
  SUM(COUNT(*)) OVER (PARTITION BY c.cod_red ORDER BY EXTRACT(YEAR FROM u.date_create), EXTRACT(MONTH FROM u.date_create)) AS cantidad_acumulada
FROM
  usuario u
LEFT JOIN paciente p ON u.id_usuario = p.id_paciente
LEFT JOIN persona per ON u.id_usuario = per.id_persona
LEFT JOIN centro c ON c.cen_asi_cod = p.cod_centro
LEFT JOIN red r ON r.cod_red = c.cod_red
WHERE
  TO_DATE(per.fecha_nacimiento, 'DD/MM/YYYY') <= (CURRENT_DATE - INTERVAL '18 years')
GROUP BY
  EXTRACT(YEAR FROM u.date_create),
  EXTRACT(MONTH FROM u.date_create),
  c.cod_red
ORDER BY
  EXTRACT(YEAR FROM u.date_create),
  EXTRACT(MONTH FROM u.date_create),
  c.cod_red
"""

usuarios_miconsulta = pd.read_sql_query(usuarios_miconsulta_query, con=connection1)


In [16]:
usuarios_miconsulta.to_sql(name=f'usuarios_miconsulta', con=connection3, if_exists='replace', index=False,chunksize=10000)

89

In [17]:
usuarios_miconsulta

,año,mes,cod_red,cantidad_acumulada
0,2021.0,11.0,05,13.0
1,2021.0,11.0,06,25.0
2,2021.0,11.0,07,13.0
3,2021.0,11.0,23,1.0
4,2021.0,12.0,05,308.0
...,...,...,...,...
1084,2024.0,9.0,35,2623.0
1085,2024.0,9.0,36,1697.0
1086,2024.0,9.0,40,2491.0
1087,2024.0,9.0,99,256.0


In [18]:
usuarios_solcita_query="""SELECT
  EXTRACT(YEAR FROM date_create) AS año,
  EXTRACT(MONTH FROM date_create) AS mes,
  COUNT(DISTINCT numero_doc_ident) AS cantidad_personas,
  cen_asi_cod AS centro
FROM
  cita
GROUP BY
  EXTRACT(YEAR FROM date_create),
  EXTRACT(MONTH FROM date_create),
  cen_asi_cod
ORDER BY
  EXTRACT(YEAR FROM date_create),
  EXTRACT(MONTH FROM date_create),
  cen_asi_cod
"""

usuarios_solcita = pd.read_sql_query(usuarios_solcita_query, con=connection1)


In [19]:
usuarios_solcita.to_sql(name=f'usuarios_solcita', con=connection3, if_exists='replace', index=False,chunksize=10000)

468

In [20]:
connection1.close()
connection2.close()
engine1.dispose()
engine2.dispose()